# Experiment 035: batch 8 checkpointing pilot (Colab)

Zero-contact pilot keeps batch 8 and accumulation 1, enables gradient checkpointing and stops after 30 updates. The pilot weights are discarded and no evaluation runs

## 0. Compute units and Drive

In [1]:
import glob, json, os, time
ALLOCATOR_CONFIG = 'expandable_segments:True'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = ALLOCATOR_CONFIG
session_id = globals().get('session_id', time.strftime('%Y%m%d_%H%M%S'))
cu_before = 484.39
unit_rate = 6.77
print('session', session_id, '| cu_before =', cu_before if cu_before is not None else 'FILL ME')

from google.colab import drive
drive.mount('/content/drive')
ROOT = '/content/drive/MyDrive/inm363'
RESULTS = os.path.join(ROOT, 'results')
OPEN_UNITS = os.path.join(RESULTS, 'open_cu_sessions')
UNITS_LOG = os.path.join(RESULTS, 'colab_units.jsonl')
os.makedirs(OPEN_UNITS, exist_ok=True)

old_sessions = [path for path in glob.glob(os.path.join(OPEN_UNITS, '*.json'))
                if not path.endswith(f'{session_id}.json')]
open_path = os.path.join(OPEN_UNITS, f'{session_id}.json')
if cu_before is None or unit_rate is None:
    print('fill cu_before and unit_rate, then rerun this cell')
else:
    for old_path in old_sessions:
        with open(old_path) as f:
            old = json.load(f)
        old['ts_end'] = time.strftime('%Y-%m-%d %H:%M')
        old['colab_cu_after'] = cu_before
        old['colab_cu_used'] = round(old['colab_cu_before'] - cu_before, 2)
        old['closed_on_resume'] = True
        with open(UNITS_LOG, 'a') as f:
            f.write(json.dumps(old) + '\n')
        os.remove(old_path)
        print('closed previous session:', old['session_id'])
    if not os.path.exists(open_path):
        record = {'session_id': session_id, 'ts_start': time.strftime('%Y-%m-%d %H:%M'),
                  'notebook': '035_1_colab_checkpointing_pilot',
                  'colab_cu_before': cu_before, 'unit_rate': unit_rate}
        with open(open_path, 'w') as f:
            json.dump(record, f, indent=2)
        print('saved open session:', session_id)

session 20260818_094823 | cu_before = 484.39
Mounted at /content/drive
closed previous session: 20260817_175921
saved open session: 20260818_094823


## 1. Runtime

In [2]:
import shutil, subprocess, torch
gpu = subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True).stdout
total_gib = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(gpu.strip())
print(f'total vram: {total_gib:.1f} GiB')
assert 'A100' in gpu and total_gib >= 79, 'need the A100 80GB runtime'
assert os.environ.get('PYTORCH_CUDA_ALLOC_CONF') == ALLOCATOR_CONFIG
assert shutil.disk_usage('/content').free / 1024**3 >= 80, 'need at least 80 GiB free locally'
print('local free:', round(shutil.disk_usage('/content').free / 1024**3, 1), 'GiB')
print('drive free:', round(shutil.disk_usage(ROOT).free / 1024**3, 1), 'GiB')

GPU 0: NVIDIA A100-SXM4-80GB (UUID: GPU-80f9ee60-281e-c46f-29e2-3d4e0069f33f)
total vram: 79.3 GiB
local free: 188.6 GiB
drive free: 179.2 GiB


## 2. Repository and dependencies

In [3]:
import base64
import getpass
import glob
import os
from pathlib import Path
import subprocess

REPO_SLUG = 'hastov-lab/rl-grpo-or-sft-for-llms'
repo_url = f'https://github.com/{REPO_SLUG}.git'
repo_root = Path('/content') / REPO_SLUG.split('/')[-1]


def repo_git(*args):
    git = ['git']
    env = None
    try:
        from google.colab import userdata
    except ImportError:
        userdata = None

    # Private forks can use a read token from Colab Secrets.
    if userdata is not None and REPO_SLUG != 'hastov-lab/rl-grpo-or-sft-for-llms':
        try:
            token = userdata.get('GH_TOKEN')
        except Exception:
            token = None
        if isinstance(token, dict):
            token = token.get('value')
        if not isinstance(token, str) or not token.strip():
            token = getpass.getpass('GitHub token: ')
        if not token.strip():
            raise RuntimeError('This private fork requires a GitHub token with read access.')

        # Keep the token out of the command line and the saved remote URL.
        auth = base64.b64encode(f'x-access-token:{token.strip()}'.encode()).decode()
        env = dict(os.environ, REPO_GIT_AUTH=f'Authorization: Basic {auth}')
        git.append('--config-env=http.https://github.com/.extraHeader=REPO_GIT_AUTH')

    subprocess.run([*git, *args], env=env, check=True)


if repo_root.exists():
    if not (repo_root / '.git').exists():
        raise RuntimeError(f'Expected a Git checkout at {repo_root}.')
    origin = subprocess.check_output(
        ['git', '-C', str(repo_root), 'config', '--local', '--get', 'remote.origin.url'], text=True
    ).strip()
    if origin != repo_url:
        raise RuntimeError(f'The checkout at {repo_root} belongs to a different repository.')
    branch = subprocess.check_output(
        ['git', '-C', str(repo_root), 'branch', '--show-current'], text=True
    ).strip()
    if branch != 'main':
        raise RuntimeError(f'Expected the main branch at {repo_root}.')
    repo_git('-C', str(repo_root), 'pull', '--ff-only', 'origin', 'main')
else:
    repo_git('clone', '--branch', 'main', '--single-branch', '--depth', '1', repo_url, str(repo_root))

# Use the scripts from this checkout even if another experiment is open.
code_root = repo_root / '001_Code'
if not (code_root / 'scripts' / 'run.py').is_file():
    raise RuntimeError(f'Experiment scripts are missing from {code_root}.')
os.chdir(code_root)
print('working in:', os.getcwd())

working in: /content/INM363-IndividualProject-Dissertation/001_Code


In [4]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'trl==1.7.1', 'transformers==5.5.4', 'peft==0.19.1',
                'accelerate==1.14.0', 'datasets==4.8.4', 'wandb', 'pyyaml'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-q', '-y', 'torchao'], check=True)
print('dependencies installed')

dependencies installed


## 3. Wiring checks

In [5]:
import subprocess, sys
subprocess.run([
    sys.executable, '-m', 'pytest', '-q',
    'tests/test_035_matched_grpo.py',
    'tests/test_training_wrapper_contracts.py',
    'tests/test_checkpoint.py',
    'tests/test_sentinel.py',
    'tests/test_protocol_freeze.py',
], check=True)
print('wiring checks passed')

wiring checks passed


In [6]:
import json, os, shutil, sys, time
sys.path.insert(0, os.getcwd())

from config import load_config
from file_utils import read_json
from scripts.grpo_worker import stream_subprocess
from scripts.s35_matched_grpo import PILOT_RUNGS_035, bind_start, config_path

PILOT_RUNG = 'A_GC'
TECH = os.path.join(ROOT, 'results', '035_technical')
start_path = os.path.join(TECH, 'starts', 'B3', 'seed0.json')
cfg = load_config(config_path('B3', 0, 'full'))
start_record = bind_start(cfg, start_path)
print('start verified:', start_record['source_tag'])

start verified: 003_COLAB_FULL_B3_sft_grpo_seed0


## 4. Technical pilot

In [ ]:
pilot_dir = os.path.join(TECH, 'pilots', 'B3', PILOT_RUNG)
report_path = os.path.join(pilot_dir, 'pilot_report.json')
status_path = os.path.join(pilot_dir, 'worker_status.json')
precision_path = os.path.join(pilot_dir, 'precision_report.json')
local_out = '/content/035_pilot_B3_A_GC'
os.makedirs(pilot_dir, exist_ok=True)

if os.path.exists(report_path):
    report = read_json(report_path)
    print('existing pilot report:', report_path)
else:
    if os.path.isdir(local_out):
        shutil.rmtree(local_out)
    if os.path.exists(status_path):
        os.remove(status_path)
    command = [
        sys.executable, 'scripts/s35_worker.py', '--task', 'pilot',
        '--config', config_path('B3', 0, 'full'),
        '--start-record', start_path, '--rung', PILOT_RUNG,
        '--status', status_path, '--out-dir', local_out,
        '--result-dir', pilot_dir, '--pilot-report', report_path,
        '--precision-report', precision_path,
    ]
    started = time.time()
    try:
        process = stream_subprocess(command)
    finally:
        if os.path.isdir(local_out):
            shutil.rmtree(local_out)
    status = read_json(status_path) if os.path.exists(status_path) else {
        'completed': False, 'error_type': 'MissingStatus'}
    status['parent_wall_seconds'] = round(time.time() - started, 1)
    with open(os.path.join(pilot_dir, '035_attempts.jsonl'), 'a') as f:
        f.write(json.dumps(status) + '\n')
    with open(os.path.join(RESULTS, 'compute_ledger.jsonl'), 'a') as f:
        f.write(json.dumps({
            'run_tag': '035_pilot_B3_A_GC', 'condition': '035_pilot',
            'timings_sec': {'worker': status['parent_wall_seconds']},
            'train_sec': status['parent_wall_seconds'],
            'train_steps': status.get('completed_updates', 0),
            'completed': bool(status.get('completed')), 'gpu': status.get('gpu'),
        }) + '\n')
    if not os.path.exists(report_path):
        raise RuntimeError(f'pilot stopped without a technical verdict: {status}')
    report = read_json(report_path)
    if process.returncode != 0 and not status.get('cuda_oom'):
        raise RuntimeError(f'pilot failed an integrity gate: {status}')

assert report.get('scale') == 'B3' and report.get('seed') == 0
assert report.get('rung') == PILOT_RUNG
assert report.get('execution') == PILOT_RUNGS_035[PILOT_RUNG]
assert report.get('allocator_config') == ALLOCATOR_CONFIG

Streaming output truncated to the last 5000 lines.
│ │ inside        │               │               │              │           │ │
│ │ <answer></an… │               │               │              │           │ │
│ │ tags.         │               │               │              │           │ │
│ │               │               │               │              │           │ │
│ │ Wolf defines  │               │               │              │           │ │
│ │ a “moral      │               │               │              │           │ │
│ │ saint” as a   │               │               │              │           │ │
│ │ person who:   │               │               │              │           │ │
│ │ A. is as      │               │               │              │           │ │
│ │ morally       │               │               │              │           │ │
│ │ worthy as     │               │               │              │           │ │
│ │ possible.     │               │               │       

In [ ]:
print('rung:', PILOT_RUNG)
print('completed updates:', report.get('completed_updates'))
print('peak reserved GiB:', report.get('peak_reserved_gib'))
print('peak reserved fraction:', round(report.get('peak_reserved_fraction', 0), 4))
print('passed:', report.get('passed'))
if report.get('passed'):
    print('pilot passed; the scientific run still needs a separate plan amendment')
else:
    print('pilot did not pass; do not start a scientific arm from this notebook')

rung: A_GC
completed updates: 30
peak reserved GiB: 67.22
peak reserved fraction: 0.8482
passed: True
pilot passed; the scientific run still needs a separate plan amendment


## 5. Close compute units

In [ ]:
cu_after = None
print('cu_after =', cu_after if cu_after is not None else 'FILL ME')
if cu_after is None:
    print('fill cu_after, then rerun this cell')
elif not os.path.exists(open_path):
    print('no open session record')
else:
    with open(open_path) as f:
        record = json.load(f)
    record['ts_end'] = time.strftime('%Y-%m-%d %H:%M')
    record['colab_cu_after'] = cu_after
    record['colab_cu_used'] = round(record['colab_cu_before'] - cu_after, 2)
    record['completed'] = True
    with open(UNITS_LOG, 'a') as f:
        f.write(json.dumps(record) + '\n')
    os.remove(open_path)
    print('logged:', record)

cu_after = FILL ME
fill cu_after, then rerun this cell
